# The data

Two things go into this tutorial: the **positions and dates** of real drifting instruments in the Arctic, and the
**output of a climate model** sampled at those positions and dates. Together they make a set of *synthetic observations*
of sea-ice velocity — and because the "world" they come from is a model, we also know what the velocity was everywhere
else, which is what lets us score a reconstruction honestly later on.

```{admonition} Running the code on this page
:class: tip
Every page of this book is a notebook. Click the **rocket icon** at the top → **Live Code** to run and edit the cells
right here (the first start can take a few minutes). Once the kernel is ready, click **Restart & Run All** next to the
status line *before* editing anything: the cells share one kernel and later ones need the imports from the first.
Then change a cell and press its *Run* button. More in [How to use this book](intro.md#how-to-use-this-book).
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments, teaching

## Ice-Tethered Profilers

An **Ice-Tethered Profiler (ITP)** is an instrument system frozen into an ice floe: a surface buoy on the ice with a
GPS and satellite link, and a profiler that crawls up and down a wire hanging below, measuring temperature and salinity
in the upper ocean. The floe drifts with the ice, so an ITP samples a *track* across the Arctic over months to years.
About a hundred have been deployed by the Woods Hole Oceanographic Institution since 2004
([whoi.edu/itp](https://www.whoi.edu/itp)).

For this tutorial we use **only the positions**: one GPS fix per ITP per day. Where an ITP was tells us *where and when*
an observing system was sampling, and the daily displacement of a drifting buoy is, in fact, how sea-ice velocity is
measured in reality. The map below shows every track with at least 30 days of data, labelled by its id in our dataset.

In [ ]:
plots.tracks(min_days=30);

Most tracks spiral around the **Beaufort Gyre** (the clockwise circulation north of Alaska and Canada, where most ITPs
have been deployed); a few cross the central Arctic and exit through **Fram Strait** east of Greenland in the
Transpolar Drift. Keep this picture in mind: the observing system is very unevenly distributed, and that unevenness is
exactly what the last page of this book is about.

## Synthetic observations from a climate model

Real ITPs do not measure wind, and we would have no way to check a reconstruction of ice velocity in places where no
instrument was. So we build a **testbed**: we take the output of a climate model — IPSL-CM6A-LR, a CMIP6 model with a
1° ocean/sea-ice grid — and *sample it where the ITPs were*, on the same calendar days. At each ITP position and day we
record the model's

* 10-m wind (`uas`, `vas`),
* sea-ice concentration (`sic`),
* sea-ice velocity (`siu`, `siv`).

This is done for **31 ensemble members** of the model's historical simulation: 31 runs of the same model with the same
forcing but different weather, so the same ITP track yields 31 different realisations of wind and ice. That multiplies
the data and, as we will see, gives a clean way to split it.

The whole dataset is one 7 MB NetCDF file that ships with the repository. The `obs_*` variables are the synthetic
observations (dimensions `member` × `sample`); the `map_*` variables are full-Arctic maps from one member, used later.

In [ ]:
data.ds

The maps below show the synthetic observations of one member along the tracks: when each sample was taken (colour by
day of year — the tracks are continuous in time), and what the model says about ice concentration, ice speed and
wind speed there.

In [ ]:
plots.obs_overview(member="r1i1p1f1")

```{note}
*How the file was made* — download of the ITP positions and the model output, co-location, and one subtlety: the model
stores ice velocity along its own curvilinear grid axes, which are rotated by tens of degrees from east/north over the
Arctic, so the velocities are rotated to geographic components before use. Scripts and details are in
[`preprocessing/`](https://github.com/lahoffman/ml-tutorial-OAISIS/tree/main/preprocessing) of the repository; nothing
there needs to be run for the tutorial.
```

## The same fields everywhere: the model maps

The point of the exercise is to predict ice velocity *where there are no ITPs*. For that we need the inputs on the
whole grid — and in the testbed we also have the true ice velocity on the whole grid to compare with. The dataset
carries ten days of full-Arctic maps from member `r1i1p1f1` (which is why that member is kept out of training later).

Below: the quantity we want to predict (ice velocity: speed as colour, direction as arrows), the two physical inputs
(wind, ice concentration), and where the ITPs sampled. The ice velocity field is clearly organised by the wind field —
the relationship the network will have to learn — but not identical to it: the ice also responds to its concentration,
the coast and the ocean.

In [ ]:
plots.input_maps(t=0)

```{admonition} Try it
:class: tip
In *Live Code* mode, change `t=0` to another day (`0`–`9`), or look at a different member in `plots.obs_overview`
(`"r2i1p1f1"` … `"r31i1p1f1"`): same tracks, different weather.
```

Next: [turning these samples into a machine-learning dataset](02_ml_dataset.ipynb).